# Test to Speech Models analysis

This experiment evaluates the performance of text-to-speech systems for generating spoken responses within the object locator application. The objective is to identify a TTS approach that provides clear and natural speech output while maintaining low latency for real-time interaction. Since the application delivers spoken feedback after retrieving the object location, the TTS component must convert short textual responses into audible speech efficiently. The evaluation therefore focuses on balancing responsiveness and speech quality to determine the most suitable TTS solution for the voice interaction component of the system.

The evaluation will be based on the following metrics:
- Speech Generation Latency
- End-to-End Processing Time
- Speech Naturalness
- Integration Complexity

Speech generation latency measures the time required for the TTS model to begin producing speech after receiving the input text. Lower latency is important for maintaining responsive voice feedback within the application.

End-to-end processing time measures the total time taken from sending the text response to the TTS system until the speech audio is fully generated or ready for playback. This reflects the real-world delay experienced by users when the system provides spoken feedback.

Speech naturalness evaluates the clarity and natural quality of the generated speech. While the object locator system uses short responses, intelligible and natural sounding speech improves user experience and accessibility.

Integration complexity considers how easily each TTS approach can be integrated into the system architecture. This includes factors such as external dependencies, network requirements, and compatibility with the mobile application environment.

The modules evaluated in this experiment includes:
- pyttsx3
- gTTS
- edgeTTS

pyttsx3 is an offline text-to-speech library that uses the operating system's built-in speech engine. It provides a lightweight and easily deployable solution for local speech synthesis and is commonly used as a desktop approximation of device-level TTS systems.

gTTS (Google Text-to-Speech) is a cloud-based speech synthesis service that converts text into speech using Google's speech generation models. It typically produces high-quality natural voices but requires an internet connection and introduces network latency during processing.

Sample sentences are defined for later evaluation of the models. These queries represent typical system responses in a missing item assistance scenario, where the system informs the user about the location of their belongings

In [ ]:
# lis of text queries that will be used for input for the TTS models
queries = [
    "Your wallet is on the brown table",
    "Your keys are beside the laptop",
    "Your phone is on the sofa",
    "The bottle is on the kitchen counter",
    "Your bag is next to the chair"
]

## PYTTSX3

Pyttsx3 will be initialised. It converts a set of predefined text queries into speecha nd measures the time taken to generate audio for each query. The script then computes the average generation latency and the end to end time. Speech naturalness is assessed manually by listening to each output and judging how smooth and natural the voice sounds.

In [ ]:
import time
import pandas as pd
import pyttsx3

# initialize TTS engine
engine = pyttsx3.init()

results = []

# perform the warm up run using the first query
# this also makes the timing more consistent
engine.say(queries[0])
engine.runAndWait()
# looping through each text query to evaluate the TTS performance
for text in queries:

    start_time = time.time()
    # convert text to speech and then play it
    engine.say(text)
    engine.runAndWait()

    end_time = time.time()

    latency = end_time - start_time
    # storing the result for this query
    results.append({
        "text": text,
        "latency_sec": latency
    })

df = pd.DataFrame(results)

# Compute average latency across all queries
avg_latency = df["latency_sec"].mean()

# end to end time (same as latency in this TTS experiment)
end_to_end_time = avg_latency

# storing final evaluation metrics in the pandas dataframe
results_df = pd.DataFrame([{
    "Model": "pyttsx3",
    "Speech Generation Latency (sec)": avg_latency,
    "End-to-End Processing Time (sec)": end_to_end_time
}])


In [3]:
results_df

,Model,Speech Generation Latency (sec),End-to-End Processing Time (sec)
0,pyttsx3,0.162222,0.162222


## GTTS

The gTTS model is initialized and evaluated using the same evaluation pipeline as pyttsx3.

In [ ]:
import time
import pandas as pd
from gtts import gTTS
import os

results = []

# warm up run for consistent timing calculation
tts = gTTS(queries[0])
# save it so the voice can be saved and then manually evaluated
tts.save("warmup.mp3")
# looping through each query
for i, text in enumerate(queries):

    start_time = time.time()
    # converting to text to speech and then save it 
    tts = gTTS(text)
    filename = f"gtts_output_{i}.mp3"
    tts.save(filename)

    end_time = time.time()
    # latency calculation for each query
    latency = end_time - start_time

    results.append({
        "text": text,
        "latency_sec": latency
    })

    # delete file so disk caching does not affect results
    os.remove(filename)

df = pd.DataFrame(results)

# compute metrics
avg_latency = df["latency_sec"].mean()

# end to end time (same as latency for this experiment)
end_to_end_time = avg_latency
new_row = pd.DataFrame([{
    "Model": "gTTS",
    "Speech Generation Latency (sec)": avg_latency,
    "End-to-End Processing Time (sec)": end_to_end_time
}])

try:
    results_df = pd.concat([results_df, new_row], ignore_index=True)
except NameError:
    results_df = new_row

In [5]:
results_df

,Model,Speech Generation Latency (sec),End-to-End Processing Time (sec)
0,pyttsx3,0.162222,0.162222
1,gTTS,0.227579,0.227579


## Edge TTS


The edgeTTS model is initialized and evaluated using the same evaluation pipeline as pyttsx3. However, since the edge-tts library operates using asynchronous API calls, the evaluation is implemented using async functions.

In [ ]:
import time
import pandas as pd
import os
import asyncio
import edge_tts

results = []
# async function
async def run_test():

    # Perform a warm-up run using the first query
    communicate = edge_tts.Communicate(queries[0], "en-US-AriaNeural")
    await communicate.save("warmup.mp3")
    # loop through each query to evaluate Edge TTS performance
    for i, text in enumerate(queries):

        start_time = time.time()
        # initialise the model with the given text and specified voice
        communicate = edge_tts.Communicate(text, "en-US-AriaNeural")
        filename = f"edge_output_{i}.mp3"
        # generating the speech asynchronously and saving to file
        await communicate.save(filename)

        end_time = time.time()
        # latency calculation for this query
        latency = end_time - start_time

        results.append({
            "text": text,
            "latency_sec": latency
        })
        # deleting generated file
        os.remove(filename)

    df = pd.DataFrame(results)
    # computing mean latency across all queries
    avg_latency = df["latency_sec"].mean()

    end_to_end_time = avg_latency
    # append the results into the existing dataframe
    new_row = pd.DataFrame([{
        "Model": "Edge TTS",
        "Speech Generation Latency (sec)": avg_latency,
        "End-to-End Processing Time (sec)": end_to_end_time
    }])

    global results_df
    try:
        results_df = pd.concat([results_df, new_row], ignore_index=True)
    except NameError:
        results_df = new_row


await run_test()

In [8]:
results_df

,Model,Speech Generation Latency (sec),End-to-End Processing Time (sec)
0,pyttsx3,0.162222,0.162222
1,gTTS,0.227579,0.227579
2,Edge TTS,0.485907,0.485907


pyttsx3 achieved the lowest latency (≈0.16 s). This is expected because pyttsx3 relies on the operating system’s built-in speech engine, allowing the text-to-speech conversion to occur entirely on the local device without any network communication. As a result, the speech generation process begins almost immediately after the text is passed to the engine.

gTTS produced slightly higher latency (≈0.23 s). Unlike pyttsx3, gTTS is a cloud-based service that requires the text input to be transmitted to Google’s servers, where the speech audio is generated and returned as an MP3 file. Although the latency increase is relatively small in this experiment, it still reflects the additional network communication and file generation steps involved in the process.

Edge TTS showed the highest latency (≈0.49 s). This is because Edge TTS uses neural speech synthesis models hosted remotely, which require additional processing time to generate more natural and expressive speech. While this results in improved speech quality, the increased computational complexity and network dependency lead to longer response times compared to local synthesis approaches.

Overall, the results demonstrate that local device-based speech synthesis provides the fastest response time, which is particularly beneficial for real-time interactive applications such as the object locator system. Although cloud-based solutions such as gTTS and Edge TTS may produce more natural-sounding speech, their reliance on network communication introduces additional latency. Therefore, device-level TTS remains the most suitable option for maintaining responsive voice feedback in the proposed system.